# SAM 3D Body articulation, stage by stage

This notebook runs the repository's production SAM3D-to-MHR components on one TOPKIDS shape and exposes the intermediate geometry needed to check the pipeline visually. Camera-utility demonstrations from the legacy prototype are intentionally omitted.

The expensive cells require a CUDA GPU and the checkpoints under `pkg/sam_3d_body/checkpoints/sam-3d-body-dinov3/`. Run the notebook from any directory; the setup cell locates the repository root.

## 1. Setup and production components

In [ ]:
import os

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import sys
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "articulate").is_dir():
            return candidate
    local_fallback = Path("/home/knpob/work/code/atm-dev")
    if (local_fallback / "pyproject.toml").is_file():
        return local_fallback
    raise RuntimeError("Could not find the atm-dev repository root.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import torch
import trimesh
from tqdm.auto import tqdm

from articulate.shape_estimator.sam3d_body_estimator import SAM3DShapeEstimator
from articulate.shape_loader.mesh_loader import MeshShapeLoader
from articulate.shape_prior.mhr_prior import MHRShapePrior
from articulate.shape_refiner.mhr_refiner import MHRShapeRefiner, chamfer_distance
from articulate.view_sampler.multi_view_sampler import MultiViewSampler

# Change to "html" for rotatable PyVista scenes. Static is easier to reproduce.
PYVISTA_BACKEND = "static"
EXAMPLE_PATH = Path("/home/knpob/work/data/shape-corr/TOPKIDS/off/kid21.off")
DEVICE = "cuda"
SEED = 0
N_ITERS_KEYPOINTS = 1000
N_ITERS_SURFACE = 1000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pv.set_jupyter_backend(PYVISTA_BACKEND)
pv.global_theme.background = "white"
pv.global_theme.font.color = "black"

MHR_PATH = PROJECT_ROOT / "pkg/sam_3d_body/checkpoints/sam-3d-body-dinov3/assets/mhr_model.pt"
CHECKPOINT_PATH = PROJECT_ROOT / "pkg/sam_3d_body/checkpoints/sam-3d-body-dinov3/model.ckpt"
SAM3D_ROOT = PROJECT_ROOT / "pkg/sam_3d_body"

required_paths = [EXAMPLE_PATH, MHR_PATH, CHECKPOINT_PATH, SAM3D_ROOT]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, f"Missing required assets: {missing}"
assert DEVICE != "cuda" or torch.cuda.is_available(), "CUDA is required for the configured SAM3D run."

print(f"Repository: {PROJECT_ROOT}")
print(f"Shape:      {EXAMPLE_PATH}")
print(f"Device:     {DEVICE}")

In [ ]:
shape_loader = MeshShapeLoader(shape_dir=str(EXAMPLE_PATH.parent))
view_sampler = MultiViewSampler(
    elevation_config="multi_3level",
    rotation_step=45.0,
    camera_dist=2.0,
    axis_order="yzx",
    camera_configs={
        "focal_length": 500.0,
        "camera_center": (256, 256),
        "h": 512,
        "w": 512,
    },
)
shape_estimator = SAM3DShapeEstimator(
    device=DEVICE,
    sam3d_pkg_path=str(SAM3D_ROOT),
    mhr_path=str(MHR_PATH),
    checkpoint_path=str(CHECKPOINT_PATH),
    detector_name="vitdet",
    fov_name="moge2",
    iou_threshold=0.7,
    retry_iou_threshold=True,
    iou_retry_step=0.1,
    min_iou_threshold=0.0,
    ransac_n_iters=3000,
    ransac_sample_size=8,
    ransac_inlier_thresh=0.02,
    ransac_min_inliers=500,
    ransac_seed=SEED,
)
shape_prior = MHRShapePrior(mhr_path=str(MHR_PATH), device=DEVICE)
shape_refiner = MHRShapeRefiner(
    device=DEVICE,
    sigma_px=5.0,
    conf_thresh=1e-2,
    triangulation_iters=2,
    trim_threshold=0.1,
    optimize_keypoints=True,
    lr_keypoints=1e-3,
    n_iters_keypoints=N_ITERS_KEYPOINTS,
    optimize_surface=True,
    n_views_surface=3,
    trim_mesh=True,
    merge_method="mean",
    lr_surface=1e-2,
    n_iters_surface=N_ITERS_SURFACE,
    bone_term_weight=1.0,
    keypoint_term_weight=0.1,
    src_to_tgt=True,
    src_subsample_num=5000,
    tgt_subsample_num=5000,
)

In [ ]:
def plot_image_grid(images, titles, *, columns=6, cmap=None, figsize_scale=3.0):
    rows = int(np.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(columns * figsize_scale, rows * figsize_scale))
    axes = np.atleast_1d(axes).ravel()
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image, cmap=cmap)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    for ax in axes[len(images):]:
        ax.axis("off")
    plt.tight_layout()
    return fig

def add_skeleton(plotter, points, bone_pairs, *, color="teal", width=4):
    points = np.asarray(points)
    for i, j in bone_pairs:
        if np.isfinite(points[[i, j]]).all():
            plotter.add_lines(points[[i, j]], color=color, width=width)

def show_overlay(mesh_target, mesh_mhr=None, *, points=None, skeletons=(), camera="xz", title=None):
    plotter = pv.Plotter()
    plotter.add_mesh(pv.wrap(mesh_target), color="white", opacity=0.35, label="target")
    if mesh_mhr is not None:
        plotter.add_mesh(pv.wrap(mesh_mhr), color="salmon", opacity=0.75, label="MHR")
    if points is not None:
        plotter.add_points(np.asarray(points), color="navy", point_size=5, render_points_as_spheres=True)
    for skeleton_points, pairs, color in skeletons:
        add_skeleton(plotter, skeleton_points, pairs, color=color)
    plotter.camera_position = camera
    if title:
        plotter.add_title(title)
    plotter.add_legend()
    return plotter.show()

def as_mhr_mesh(vertices, faces):
    if torch.is_tensor(vertices):
        vertices = vertices.detach().cpu().numpy()
    return trimesh.Trimesh(vertices=vertices, faces=faces, process=False)

def keypoint_rmse(predicted, target):
    predicted = np.asarray(predicted)
    target = np.asarray(target)
    valid = np.isfinite(predicted).all(axis=-1) & np.isfinite(target).all(axis=-1)
    return float(np.sqrt(np.mean((predicted[valid] - target[valid]) ** 2)))

## 2. Load one shape and render the production camera views

In [ ]:
shape = shape_loader.pipeline(str(EXAMPLE_PATH))
mesh = shape["mesh"]
print(f"{shape['name']}: {len(mesh.vertices):,} vertices, {len(mesh.faces):,} faces, area={mesh.area:.4f}")
show_overlay(mesh, camera="xz", title="Input shape")

In [ ]:
camera_views = view_sampler.sample_views()
view_sampling = view_sampler.pipeline(shape=shape, camera_views=camera_views)
observations = view_sampling["observations"]
assert len(observations) == len(camera_views), "At least one configured view failed to render."
print(f"Rendered {len(observations)} views: {list(observations)}")

In [ ]:
view_titles = [f"az={az:g}, el={el:g}" for az, el in observations]
plot_image_grid([view["rgb"] for view in observations.values()], view_titles)
plot_image_grid([view["depth"] for view in observations.values()], view_titles, cmap="viridis")

## 3. SAM3D inference and per-view similarity alignment

Inference is run once per rendered view and cached. Alignment is then evaluated with an IoU threshold of zero so every alignable view retains its diagnostic IoU; the production retry schedule is applied to that cache afterward.

In [ ]:
shape_estimator.init_estimator()
shape_prior.load()
shape_prior.load_from_estimator(shape_estimator)
print("SAM3D estimator and scripted MHR prior loaded.")

In [ ]:
sam3d_predictions = {}
inference_errors = {}
for view_key, view_data in tqdm(observations.items(), desc="SAM3D inference"):
    try:
        sam3d_predictions[view_key] = shape_estimator.estimate(
            img_rgb=view_data["rgb"],
            K=view_data["cam"]["K"],
        )
    except Exception as error:
        inference_errors[view_key] = repr(error)

print(f"Successful inference: {len(sam3d_predictions)}/{len(observations)}")
if inference_errors:
    print("Failed views:", inference_errors)
assert sam3d_predictions, "SAM3D did not produce a prediction for any view."

In [ ]:
prediction_panels = []
prediction_titles = []
for view_key, output in sam3d_predictions.items():
    image = observations[view_key]["rgb"].copy()
    prediction_panels.append(image)
    prediction_titles.append(f"az={view_key[0]:g}, el={view_key[1]:g}")

fig = plot_image_grid(prediction_panels, prediction_titles)
for ax, output in zip(fig.axes, sam3d_predictions.values()):
    keypoints = np.asarray(output["pred_keypoints_2d"])
    ax.scatter(keypoints[:, 0], keypoints[:, 1], s=5, marker="x", c="cyan")
fig.suptitle("Rendered observations with SAM3D 2D keypoints", y=1.01)

In [ ]:
configured_iou_threshold = shape_estimator.iou_threshold
shape_estimator.iou_threshold = 0.0
alignment_candidates = {}
alignment_errors = {}
for view_key, output in tqdm(sam3d_predictions.items(), desc="Similarity alignment"):
    try:
        result = shape_estimator.se3_align(
            mesh=mesh,
            view_data=observations[view_key],
            sam3d_output=output,
        )
        if result is not None:
            alignment_candidates[view_key] = result
    except Exception as error:
        alignment_errors[view_key] = repr(error)
shape_estimator.iou_threshold = configured_iou_threshold

current_iou = configured_iou_threshold
while current_iou >= shape_estimator.min_iou_threshold:
    results = {key: value for key, value in alignment_candidates.items() if value["iou"] >= current_iou}
    if results or not shape_estimator.retry_iou_threshold or current_iou == shape_estimator.min_iou_threshold:
        break
    current_iou = round(current_iou - shape_estimator.iou_retry_step, 1)

shape_estimator.iou_threshold = current_iou
print(f"Accepted {len(results)}/{len(alignment_candidates)} aligned views at IoU >= {current_iou:.1f}")
if alignment_errors:
    print("Alignment failures:", alignment_errors)
assert len(results) >= 2, "Fewer than two valid views remain; triangulation is not defined."

In [ ]:
keys = list(alignment_candidates)
ious = np.array([alignment_candidates[key]["iou"] for key in keys])
labels = [f"{az:g}/{el:g}" for az, el in keys]
colors = ["tab:green" if key in results else "tab:red" for key in keys]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(np.arange(len(keys)), ious, color=colors)
ax.axhline(current_iou, color="black", linestyle="--", label=f"threshold={current_iou:.1f}")
ax.set_xticks(np.arange(len(keys)), labels, rotation=45, ha="right")
ax.set_ylabel("silhouette IoU")
ax.set_xlabel("azimuth/elevation")
ax.set_title("Per-view alignment acceptance")
ax.legend()
plt.tight_layout()

In [ ]:
overlay_images = []
overlay_titles = []
for view_key, result in results.items():
    rgb = observations[view_key]["rgb"].astype(np.float32) / 255.0
    rgba = result["img_rgba_mhr"].astype(np.float32) / 255.0
    alpha = rgba[..., 3:4]
    overlay_images.append(np.clip(0.55 * rgb + 0.45 * (rgba[..., :3] * alpha + rgb * (1 - alpha)), 0, 1))
    overlay_titles.append(f"{view_key}, IoU={result['iou']:.3f}")
plot_image_grid(overlay_images, overlay_titles, columns=3)

In [ ]:
plotter = pv.Plotter()
plotter.add_mesh(pv.wrap(mesh), color="white", opacity=0.25, label="target")
for view_key, result in results.items():
    plotter.add_mesh(pv.wrap(result["mesh_mhr_aligned"]), color="salmon", opacity=0.12)
plotter.camera_position = "xz"
plotter.add_title("All accepted per-view MHR alignments")
plotter.add_legend()
plotter.show()

## 4. Robust multi-view triangulation

In [ ]:
shape_estimation = {
    "shape": shape,
    "results": results,
    "camera_configs": view_sampling["camera_configs"],
    "axis_order": view_sampling["axis_order"],
    "estimator": shape_estimator.estimator,
}
X_hat, used_views, conf_by_view = shape_refiner.joint_triangulate(
    results, camera_configs=shape_estimation["camera_configs"]
)
finite_keypoints = np.isfinite(X_hat).all(axis=1)
print(f"Finite triangulated keypoints: {finite_keypoints.sum()}/{len(X_hat)}")
print(f"Views used per keypoint: min={used_views.min()}, median={np.median(used_views):g}, max={used_views.max()}")
assert finite_keypoints.all(), "Triangulation produced non-finite keypoints; inspect confidence and accepted views."

In [ ]:
body_bones = shape_prior.get_skeleton(mode="body")
show_overlay(
    mesh,
    points=X_hat,
    skeletons=[(X_hat, body_bones, "teal")],
    camera="xz",
    title="Robustly triangulated MHR70 skeleton",
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 7), constrained_layout=True)
axes[0].bar(np.arange(len(used_views)), used_views)
axes[0].set(title="Views retained per triangulated keypoint", xlabel="MHR70 keypoint ID", ylabel="views")
im = axes[1].imshow(conf_by_view, aspect="auto", cmap="magma", vmin=0, vmax=1)
axes[1].set(title="Tukey confidence after reprojection gating", xlabel="MHR70 keypoint ID", ylabel="accepted view index")
fig.colorbar(im, ax=axes[1], label="confidence")

## 5. Optimize MHR pose against triangulated keypoints

In [ ]:
identity_initial, model_parameters_initial, expression_initial = shape_refiner.stack_mhr_params(results, DEVICE)
R_all, t_all, s_all = shape_refiner.stack_alignment_params(results, DEVICE, dtype=identity_initial.dtype)
vertices_initial, keypoints_initial, _ = shape_prior.generate(
    identity_initial, model_parameters_initial, expression_initial, R_all, s_all, t_all, return_keypoints=True
)
initial_rmse = keypoint_rmse(keypoints_initial[:, :70].detach().cpu().numpy(), np.broadcast_to(X_hat, keypoints_initial[:, :70].shape))
print(f"Initial multi-view keypoint RMSE: {initial_rmse:.6f}")

In [ ]:
identity_pose, model_parameters_pose, expression_pose = shape_refiner.pose_optim(
    results=results, X_hat=X_hat, shape_prior=shape_prior, device=DEVICE
)
vertices_pose, keypoints_pose, _ = shape_prior.generate(
    identity_pose, model_parameters_pose, expression_pose, R_all, s_all, t_all, return_keypoints=True
)
pose_rmse = keypoint_rmse(keypoints_pose[:, :70].detach().cpu().numpy(), np.broadcast_to(X_hat, keypoints_pose[:, :70].shape))
print(f"Pose-optimized multi-view keypoint RMSE: {pose_rmse:.6f}")
print(f"RMSE reduction: {initial_rmse - pose_rmse:.6f}")

In [ ]:
view_index = 0
initial_mesh = as_mhr_mesh(vertices_initial[view_index], shape_estimator.estimator.faces)
pose_mesh = as_mhr_mesh(vertices_pose[view_index], shape_estimator.estimator.faces)

plotter = pv.Plotter(shape=(1, 2), window_size=(1400, 650))
for column, (stage_name, stage_mesh, stage_keypoints) in enumerate([
    ("Before keypoint optimization", initial_mesh, keypoints_initial[view_index, :70].detach().cpu().numpy()),
    ("After keypoint optimization", pose_mesh, keypoints_pose[view_index, :70].detach().cpu().numpy()),
]):
    plotter.subplot(0, column)
    plotter.add_mesh(pv.wrap(mesh), color="white", opacity=0.25)
    plotter.add_mesh(pv.wrap(stage_mesh), color="salmon", opacity=0.65)
    add_skeleton(plotter, X_hat, body_bones, color="teal")
    add_skeleton(plotter, stage_keypoints, body_bones, color="red")
    plotter.add_title(stage_name)
    plotter.camera_position = "xz"
plotter.show()

## 6. Trim the target surface and select the best views

In [ ]:
alignment_inliers = np.concatenate(
    [result["alignment"]["points_tgt_ori"] for result in results.values()], axis=0
)
mesh_trimmed = shape_refiner.trim_mesh_by_results(mesh, results)
surface_samples, _ = trimesh.sample.sample_surface_even(mesh_trimmed, count=5000)
print(f"Original mesh: {len(mesh.vertices):,} vertices / {len(mesh.faces):,} faces")
print(f"Trimmed mesh:  {len(mesh_trimmed.vertices):,} vertices / {len(mesh_trimmed.faces):,} faces")

In [ ]:
plotter = pv.Plotter(shape=(1, 3), window_size=(1800, 600))
plotter.subplot(0, 0)
plotter.add_mesh(pv.wrap(mesh), color="white", opacity=0.25)
plotter.add_points(alignment_inliers, color="navy", point_size=2)
plotter.add_title("Accepted alignment inliers")
plotter.camera_position = "xz"
plotter.subplot(0, 1)
plotter.add_mesh(pv.wrap(mesh), color="lightgray", opacity=0.2)
plotter.add_mesh(pv.wrap(mesh_trimmed), color="cornflowerblue", opacity=0.85)
plotter.add_title("Trimmed optimization target")
plotter.camera_position = "xz"
plotter.subplot(0, 2)
plotter.add_points(surface_samples, scalars=surface_samples[:, 2], cmap="viridis", point_size=3)
plotter.add_title("5,000 sampled target points")
plotter.camera_position = "xz"
plotter.show()

In [ ]:
view_keys = list(results)
chamfer_by_view = {
    key: chamfer_distance(mesh_trimmed.vertices, vertices_pose[index].detach().cpu().numpy())
    for index, key in enumerate(view_keys)
}
selected_keys = [
    key for _, key in sorted((score, key) for key, score in chamfer_by_view.items())[:shape_refiner.n_views_surface]
]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(
    np.arange(len(view_keys)),
    [chamfer_by_view[key] for key in view_keys],
    color=["tab:green" if key in selected_keys else "tab:gray" for key in view_keys],
)
ax.set_xticks(np.arange(len(view_keys)), [f"{az:g}/{el:g}" for az, el in view_keys], rotation=45, ha="right")
ax.set(title="Chamfer-based view selection after pose optimization", xlabel="azimuth/elevation", ylabel="bidirectional squared Chamfer")
plt.tight_layout()
print("Selected views:", selected_keys)

## 7. Surface optimization and final merge

`shape_optim` is called directly here because it contains the production view-selection and parameter-restacking lifecycle before surface fitting.

In [ ]:
identity_surface, model_parameters_surface, expression_surface, results_for_surface = shape_refiner.shape_optim(
    results=results,
    mesh=mesh_trimmed,
    X_hat=X_hat,
    shape_prior=shape_prior,
    estimator=shape_estimator.estimator,
    identity_coeffs=identity_pose,
    model_parameters=model_parameters_pose,
    face_expr_coeffs=expression_pose,
    device=DEVICE,
)
assert list(results_for_surface) == selected_keys, "Production selection disagrees with the diagnostic ranking."
print("Surface optimization used:", list(results_for_surface))

In [ ]:
R_surface, t_surface, s_surface = shape_refiner.stack_alignment_params(
    results_for_surface, DEVICE, dtype=identity_surface.dtype
)
vertices_surface, keypoints_surface, _ = shape_prior.generate(
    identity_surface, model_parameters_surface, expression_surface,
    R_surface, s_surface, t_surface, return_keypoints=True,
)
mesh_merged = shape_refiner.merge_meshes(
    results=results_for_surface,
    shape_prior=shape_prior,
    identity_coeffs=identity_surface,
    model_parameters=model_parameters_surface,
    face_expr_coeffs=expression_surface,
    mesh_target=mesh_trimmed,
    estimator=shape_estimator.estimator,
    merge_method=shape_refiner.merge_method,
    device=DEVICE,
)

surface_chamfers = [
    chamfer_distance(mesh_trimmed.vertices, vertices_surface[i].detach().cpu().numpy())
    for i in range(len(results_for_surface))
]
final_chamfer = chamfer_distance(mesh_trimmed.vertices, mesh_merged.vertices)
surface_rmse = keypoint_rmse(
    keypoints_surface[:, :70].detach().cpu().numpy(),
    np.broadcast_to(X_hat, keypoints_surface[:, :70].shape),
)
print(f"Selected-view surface Chamfer: {[round(value, 6) for value in surface_chamfers]}")
print(f"Final merged Chamfer:          {final_chamfer:.6f}")
print(f"Final selected-view KP RMSE:   {surface_rmse:.6f}")

In [ ]:
selected_view_index = view_keys.index(selected_keys[0])
stage_meshes = [
    ("Initial estimate", as_mhr_mesh(vertices_initial[selected_view_index], shape_estimator.estimator.faces)),
    ("Pose refined", as_mhr_mesh(vertices_pose[selected_view_index], shape_estimator.estimator.faces)),
    ("Surface refined", as_mhr_mesh(vertices_surface[0], shape_estimator.estimator.faces)),
    ("Final mean merge", mesh_merged),
]
plotter = pv.Plotter(shape=(1, 4), window_size=(2000, 520))
for column, (stage_name, stage_mesh) in enumerate(stage_meshes):
    plotter.subplot(0, column)
    plotter.add_mesh(pv.wrap(mesh), color="white", opacity=0.25)
    plotter.add_mesh(pv.wrap(stage_mesh), color="salmon", opacity=0.7)
    plotter.add_title(stage_name)
    plotter.camera_position = "xz"
plotter.show()

In [ ]:
plotter = pv.Plotter(shape=(1, 3), window_size=(1800, 600))
for column, camera in enumerate(["xy", "xz", "zy"]):
    plotter.subplot(0, column)
    plotter.add_mesh(pv.wrap(mesh), color="white", opacity=0.3, label="input")
    plotter.add_mesh(pv.wrap(mesh_merged), color="salmon", opacity=0.7, label="merged MHR")
    plotter.add_title(f"Final overlay: {camera}")
    plotter.camera_position = camera
plotter.add_legend()
plotter.show()

In [ ]:
diagnostics = {
    "shape": shape["name"],
    "rendered_views": len(observations),
    "successful_inference_views": len(sam3d_predictions),
    "accepted_alignment_views": len(results),
    "effective_iou_threshold": current_iou,
    "mean_accepted_iou": float(np.mean([result["iou"] for result in results.values()])),
    "finite_triangulated_keypoints": int(finite_keypoints.sum()),
    "initial_keypoint_rmse": initial_rmse,
    "pose_keypoint_rmse": pose_rmse,
    "surface_keypoint_rmse": surface_rmse,
    "selected_surface_views": list(results_for_surface),
    "final_merged_chamfer": final_chamfer,
}
diagnostics

## Visual correctness checklist

- Rendered silhouettes should remain centered and cover the body from every elevation.
- SAM3D keypoints should lie on the rendered person, with left/right limbs consistent across views.
- Accepted aligned MHR meshes should share the target's scale, orientation, and location.
- The triangulated skeleton should lie inside the target mesh without mirrored or swapped limbs.
- Keypoint optimization should reduce RMSE and pull the red skeleton toward the teal skeleton.
- Trimming should remove unsupported target geometry without cutting away observed body regions.
- Selected views should have the lowest displayed Chamfer scores.
- Surface optimization and merging should improve coverage without collapsing limbs or changing transform direction.